Final workable with searchable tab work in google colab and also download in pc folder and its files view ,  tree view

In [12]:
# GOOGLE DRIVE FULL TREE + LIVE SEARCH + AUTO DOWNLOAD
# 100% WORKING – TESTED RIGHT NOW IN COLAB

from googleapiclient.discovery import build
from google.colab import auth, drive, files
from google.auth import default
from IPython.display import display, HTML
import time

# Setup
drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
service = build('drive', 'v3', credentials=creds)

output_file = "/content/drive/MyDrive/Google_Drive_Tree_Searchable.html"
print("Scanning your Google Drive...")

# Fetch all files
def get_all():
    items = []
    token = None
    while True:
        r = service.files().list(
            q="trashed = false",
            spaces='drive',
            fields='nextPageToken, files(id,name,mimeType,parents,size)',
            pageSize=1000,
            pageToken=token
        ).execute()
        items += r.get('files', [])
        token = r.get('nextPageToken')
        if not token: break
    return items

all_items = get_all()
print(f"Total items: {len(all_items)}")

# Build tree structure
by_id = {i['id']: i for i in all_items}
children = {}
for i in all_items:
    for p in i.get('parents', []):
        children.setdefault(p, []).append(i)

roots = [i for i in all_items if not i.get('parents') or i['parents'][0] not in by_id]

def sort_kids(items):
    folders = [x for x in items if x['mimeType'] == 'application/vnd.google-apps.folder']
    files   = [x for x in items if x['mimeType'] != 'application/vnd.google-apps.folder']
    folders.sort(key=lambda x: x['name'].lower())
    files.sort(key=lambda x: x['name'].lower())
    return folders + files

# Build tree HTML
def build(item, pre="", last=True):
    name = item['name']
    folder = item['mimeType'] == 'application/vnd.google-apps.folder'
    icon = "Folder" if folder else "File"
    size = ""
    if not folder and 'size' in item:
        s = int(item['size']) / 1024
        size = f" ({s/1024:.1f} MB)" if s >= 1024 else f" ({s:.0f} KB)"
    conn = "└── " if last else "├── "
    line = f"{pre}{conn}{icon} <span class='n'>{name}</span><span class='s'>{size}</span><br>"
    kids = sort_kids(children.get(item['id'], []))
    for idx, k in enumerate(kids):
        line += build(k, pre + ("    " if last else "│   "), idx == len(kids)-1)
    return line

tree_html = ""
for i, r in enumerate(sort_kids(roots)):
    if r['id'] == 'root' or r.get('name') == 'My Drive':
        tree_html += "My Drive<br>"
    else:
        tree_html += f"Shared Drive: {r['name']}<br>"
    tree_html += build(r, "", i == len(roots)-1)

# FINAL HTML – using .format() to avoid any parsing issues
html_template = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Google Drive Tree + Search</title>
<style>
  body {{font-family:Arial,sans-serif;background:#f9f9f9;padding:20px;margin:0;}}
  h1 {{color:#1967d2;text-align:center;}}
  input {{width:100%;padding:15px;font-size:20px;border:3px solid #1967d2;border-radius:12px;margin:15px 0;}}
  .info {{text-align:center;font-size:18px;color:#555;}}
  .tree {{background:white;padding:25px;border-radius:12px;box-shadow:0 6px 20px rgba(0,0,0,0.1);
          font-family:Consolas,monospace;font-size:15px;line-height:1.7;overflow-x:auto;white-space:pre;}}
  .hl {{background:#ffeb3b !important;padding:2px 5px;border-radius:4px;font-weight:bold;}}
  .s {{color:#777;font-size:0.9em;}}
</style>
</head>
<body>
<h1>Google Drive Full Tree View</h1>
<input type="text" placeholder="Live Search..." autofocus>
<div class="info">Total items: {total} • {date}</div>
<div class="tree">{tree}</div>

<script>
  const inp = document.querySelector('input');
  const names = document.querySelectorAll('.n');
  const info = document.querySelector('.info');
  inp.oninput = function() {{
    const term = inp.value.toLowerCase();
    let found = 0;
    names.forEach(el => {{
      if (term && el.textContent.toLowerCase().includes(term)) {{
        el.classList.add('hl');
        found++;
      }} else {{
        el.classList.remove('hl');
      }}
    }});
    info.textContent = term ? "Found " + found + " result(s) for '" + term + "'" : "Total items: {total}";
    if (found > 0) document.querySelector('.hl')?.scrollIntoView({{behavior:'smooth', block:'center'}});
  }};
</script>
</body>
</html>"""

final_html = html_template.format(
    total=len(all_items),
    date=time.strftime("%d %b %Y, %I:%M %p"),
    tree=tree_html
)

# Save + Auto download + Show in Colab
with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_html)

files.download(output_file)      # Download start ho jayega
display(HTML(final_html))        # Live preview neeche dikhega

print("\nHO GAYA BHAI!")
print("File saved →", output_file)
print("Download + Live search dono kaam kar rahe hain!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Scanning your Google Drive...
Total items: 103


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


HO GAYA BHAI!
File saved → /content/drive/MyDrive/Google_Drive_Tree_Searchable.html
Download + Live search dono kaam kar rahe hain!
